# Financial Data Pipeline & Fundamental Analysis

## Objetivo
Construir un pipeline completo de datos financieros:

1. Extracción desde API (Financial Modeling Prep)
2. Transformación en Pandas
3. Carga en base de datos MySQL
4. Consulta estructurada
5. Cálculo de métricas financieras clave
6. Análisis fundamental del negocio

---

Este notebook representa la versión limpia y estructurada del pipeline.

In [1]:
# =========================
# 1. Setup & Configuration
# =========================

import os
import sys
import requests
import pandas as pd
from dotenv import load_dotenv

sys.path.append("../src")
from functions import create_connection

load_dotenv("../.env")

API_KEY = os.getenv("FMP_API_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

In [2]:
def extract_income_statement(symbol, api_key):
    url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={api_key}"
    
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"API Error: {response.status_code}")
    
    data = response.json()
    return pd.DataFrame(data)

In [3]:
df_raw = extract_income_statement("AAPL", API_KEY)
df_raw.head()

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,...,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-27,AAPL,USD,0000320193,2025-10-31,2025-10-31 06:01:26,2025,FY,416161000000,220960000000,...,112010000000,0,0,112010000000,0,112010000000,7.49,7.46,14948500000,15004697000
1,2024-09-28,AAPL,USD,0000320193,2024-11-01,2024-11-01 06:01:36,2024,FY,391035000000,210352000000,...,93736000000,0,0,93736000000,0,93736000000,6.11,6.08,15343783000,15408095000
2,2023-09-30,AAPL,USD,0000320193,2023-11-03,2023-11-02 18:08:27,2023,FY,383285000000,214137000000,...,96995000000,0,0,96995000000,0,96995000000,6.16,6.13,15744231000,15812547000
3,2022-09-24,AAPL,USD,0000320193,2022-10-28,2022-10-27 18:01:14,2022,FY,394328000000,223546000000,...,99803000000,0,0,99803000000,0,99803000000,6.15,6.11,16215963000,16325819000
4,2021-09-25,AAPL,USD,0000320193,2021-10-29,2021-10-28 18:04:28,2021,FY,365817000000,212981000000,...,94680000000,0,0,94680000000,0,94680000000,5.67,5.61,16701272000,16864919000


In [4]:
def transform_income_data(df_raw):
    columns_selected = [
        "fiscalYear",
        "revenue",
        "costOfRevenue",
        "grossProfit",
        "operatingIncome",
        "netIncome",
        "eps"
    ]
    
    df = df_raw[columns_selected].copy()
    df = df.sort_values("fiscalYear")
    df.reset_index(drop=True, inplace=True)
    
    return df

In [5]:
df = transform_income_data(df_raw)
df.head()

,fiscalYear,revenue,costOfRevenue,grossProfit,operatingIncome,netIncome,eps
0,2021,365817000000,212981000000,152836000000,108949000000,94680000000,5.67
1,2022,394328000000,223546000000,170782000000,119437000000,99803000000,6.15
2,2023,383285000000,214137000000,169148000000,114301000000,96995000000,6.16
3,2024,391035000000,210352000000,180683000000,123216000000,93736000000,6.11
4,2025,416161000000,220960000000,195201000000,133050000000,112010000000,7.49


In [6]:
def load_to_mysql(df, symbol):

    connection = create_connection(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME
    )

    cursor = connection.cursor()

    cursor.execute("SELECT id FROM companies WHERE symbol = %s", (symbol,))
    result = cursor.fetchone()

    if result is None:
        cursor.execute(
        "INSERT INTO companies (symbol, name, sector) VALUES (%s, %s, %s)",
        (symbol, symbol, "Unknown")
        )
        connection.commit()

        cursor.execute("SELECT id FROM companies WHERE symbol = %s", (symbol,))
        result = cursor.fetchone()

    company_id = result[0]
    
    
    insert_query = """
    INSERT INTO income_statements (
        company_id,
        fiscal_year,
        revenue,
        cost_of_revenue,
        gross_profit,
        operating_income,
        net_income,
        eps
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        revenue = VALUES(revenue),
        cost_of_revenue = VALUES(cost_of_revenue),
        gross_profit = VALUES(gross_profit),
        operating_income = VALUES(operating_income),
        net_income = VALUES(net_income),
        eps = VALUES(eps);
    """
    
    for _, row in df.iterrows():
        cursor.execute(insert_query, (
            company_id,
            row["fiscalYear"],
            row["revenue"],
            row["costOfRevenue"],
            row["grossProfit"],
            row["operatingIncome"],
            row["netIncome"],
            row["eps"]
        ))
    
    connection.commit()
    connection.close()

In [7]:
connection = create_connection(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

query = """
SELECT c.symbol, COUNT(*) as records_loaded
FROM income_statements i
JOIN companies c ON i.company_id = c.id
GROUP BY c.symbol;
"""

pd.read_sql(query, connection)

/var/folders/qz/lqkm42d57sxb3jcp56ssgksm0000gn/T/ipykernel_54371/1022634123.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, connection)


,symbol,records_loaded
0,AAPL,5
1,MSFT,5


In [8]:
load_to_mysql(df, "AAPL")

In [9]:
def run_pipeline(symbol):
    df_raw = extract_income_statement(symbol, API_KEY)
    df = transform_income_data(df_raw)
    load_to_mysql(df, symbol)
    print(f"[INFO] Pipeline executed successfully for {symbol}")

In [10]:
run_pipeline("AAPL")
run_pipeline("MSFT")

[INFO] Pipeline executed successfully for AAPL
[INFO] Pipeline executed successfully for MSFT
